# Moon phase vs crime/accident (sklearn)
Fetch Supabase data, merge moon phases with daily outcomes, and quantify relationships via linear models.

In [1]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

load_dotenv()
if not os.getenv("SUPABASE_URL") or not os.getenv("SUPABASE_KEY"):
    raise SystemExit("Missing SUPABASE_URL or SUPABASE_KEY in environment")

supabase = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_KEY"])

def fetch_table(name: str, batch_size: int = 1000, limit: int | None = None) -> pd.DataFrame:
    rows, start = [], 0
    while True:
        end = start + batch_size - 1
        resp = supabase.table(name).select("*").range(start, end).execute()
        batch = resp.data or []
        rows.extend(batch)
        if len(batch) < batch_size or (limit and len(rows) >= limit):
            break
        start += batch_size
    if limit:
        rows = rows[:limit]
    return pd.DataFrame(rows)


After fetch Function run, get all tables and store on the variables

In [3]:
crimes_df = fetch_table("chicago_crimes")
accident_df = fetch_table("chicago_accident_cleaned")
moon_df = fetch_table("cleaned_moon_data")

print(f"Crimes: {len(crimes_df)} rows")
print(f"Accidents: {len(accident_df)} rows")
print(f"Moon: {len(moon_df)} rows")

if crimes_df.empty or accident_df.empty or moon_df.empty:
    raise SystemExit("One or more tables are empty; cannot proceed")


Crimes: 251337 rows
Accidents: 110720 rows
Moon: 1826 rows


As the moon dataset has just 5 moon phases, a function to repair this column is run

In [5]:
def moon_phase_from_age(age: float) -> str:
    if age < 1.845:
        return "New Moon"
    elif age < 5.536:
        return "Waxing Crescent"
    elif age < 9.228:
        return "First Quarter"
    elif age < 12.919:
        return "Waxing Gibbous"
    elif age < 16.610:
        return "Full Moon"
    elif age < 20.302:
        return "Waning Gibbous"
    elif age < 23.993:
        return "Last Quarter"
    else:
        return "Waning Crescent"

moon_df["moon_category"] = moon_df["avg_age"].apply(moon_phase_from_age)

Before AI analysis some alterations and merges were performed. Firstly, grouping the Datasets rows daily incrementing the `crime_count` and `incidents_count`, while changing the `date_only` from the `chicago_crimes` and the `crash_date` fo the `chicago_accident_cleaned` to `date`.

Make sure that all dates are in datetime.

Next, merging the three datasets using `date` as the join key, ad droping any row missing the `avg_phase`, `moon_category`, `crime_count` and `incidents_count`

Lastly, copying the merged dataset in a `merged_temp` and adding the weekday and months to nexts steps 

In [6]:

crimes_daily = crimes_df.groupby("date_only")["crime_count"].sum().reset_index().rename(columns={"date_only": "date"})
acc_daily = accident_df.groupby("crash_date")["incidents_count"].sum().reset_index().rename(columns={"crash_date": "date"})

crimes_daily["date"] = pd.to_datetime(crimes_daily["date"])
acc_daily["date"] = pd.to_datetime(acc_daily["date"])
moon_df["date"] = pd.to_datetime(moon_df["date"])

merged = moon_df.merge(crimes_daily, on="date", how="left").merge(acc_daily, on="date", how="left")
merged = merged.dropna(subset=["avg_phase", "moon_category", "crime_count", "incidents_count"]).copy()

merged_temp = merged.copy()
merged_temp["weekday"] = merged_temp["date"].dt.day_name()
merged_temp["month"] = merged_temp["date"].dt.month_name()

After the dataset are ready, the AI analysis using the library `Sklean` step takes place.

The `X` matrix contains the variables used by `Sklean` to try to estimate the target vector `y`

The `drop_first=True` is adopted to avoid multicollinearity.

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

X = pd.get_dummies(merged["moon_category"], drop_first=True)
y = merged["crime_count"].fillna(0)

model = LinearRegression()
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n===== crime_count by category =====")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\n weights:\n", coefs)

X = pd.get_dummies(merged["avg_phase"], drop_first=True)
y = merged["crime_count"].fillna(0)

model = LinearRegression()
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n===== crime_count by phase =====")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\n weights:\n", coefs.tail(5),"\n", coefs.head(5))



===== crime_count by category =====

5-fold R^2: mean=-0.770, std=0.795

 weights:
 Waxing Crescent   -6.221735
New Moon          -6.100544
Last Quarter      -3.295011
Waxing Gibbous     0.056744
Waning Crescent    0.412315
Waning Gibbous     0.480851
Full Moon          3.354801
dtype: float64

===== crime_count by phase =====

5-fold R^2: mean=-1.247, std=0.792

 weights:
 94.25    300.0
99.55    329.0
5.23     334.0
74.17    376.0
74.25    486.0
dtype: float64 
 91.17   -285.0
20.93   -263.0
20.99   -246.0
1.97    -237.0
2.41    -230.0
dtype: float64


In order to look for others insightful findings, the linear regression is also use to try to predict the daily crimes using weekdays and months as a base

In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

X = pd.get_dummies(merged_temp[["weekday"]], drop_first=True)
y = merged_temp["crime_count"].fillna(0)

model = LinearRegression()
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n===== crime_count per weekday =====")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nWeights:\n", coefs)

X = pd.get_dummies(merged_temp[["month"]].astype("category"), drop_first=True)
y = merged_temp["crime_count"].fillna(0)

model = LinearRegression()
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n====== crime_count per month ======")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nWeights:\n", coefs)


===== crime_count per weekday =====

5-fold R^2: mean=-0.746, std=0.804

Weights:
 weekday_Thursday    -29.333506
weekday_Tuesday     -27.114756
weekday_Wednesday   -24.118662
weekday_Sunday      -20.809339
weekday_Monday      -17.970224
weekday_Saturday     -7.031128
dtype: float64

====== crime_count per month ======

5-fold R^2: mean=-0.458, std=0.769

Weights:
 month_January     -42.234409
month_February    -39.866667
month_March       -11.195699
month_December      0.439785
month_November     17.240000
month_May          38.597849
month_October      71.875269
month_August       74.894624
month_June         80.606667
month_September    85.460000
month_July         93.881720
dtype: float64


The same procedure for `crime_counts` is applied to `incidents_count`

In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

X = pd.get_dummies(merged["moon_category"], drop_first=True)
y = merged["incidents_count"].fillna(0)

model = LinearRegression()
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n=== incidents_count by category  ===")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\n weights:\n", coefs)

X = pd.get_dummies(merged["avg_phase"], drop_first=True)
y = merged["crime_count"].fillna(0)

model = LinearRegression()
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n===== incidents_count by phase =====")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\n weights:\n", coefs.tail(5),"\n", coefs.head(5))



=== incidents_count by category  ===

5-fold R^2: mean=-0.026, std=0.020

 weights:
 New Moon          -2.435972
Waxing Gibbous    -2.327584
Full Moon         -2.246701
Waning Crescent   -1.565401
Last Quarter      -1.505539
Waxing Crescent   -1.096056
Waning Gibbous     0.523817
dtype: float64

===== incidents_count by phase =====

5-fold R^2: mean=-1.247, std=0.792

 weights:
 94.25    300.0
99.55    329.0
5.23     334.0
74.17    376.0
74.25    486.0
dtype: float64 
 91.17   -285.0
20.93   -263.0
20.99   -246.0
1.97    -237.0
2.41    -230.0
dtype: float64


In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

X = pd.get_dummies(merged_temp[["weekday"]], drop_first=True)
y = merged_temp["incidents_count"].fillna(0)

model = LinearRegression()
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n===== incidents_count per weekday =====")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nWeights:\n", coefs)

X = pd.get_dummies(merged_temp[["month"]].astype("category"), drop_first=True)
y = merged_temp["incidents_count"].fillna(0)

model = LinearRegression()
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n====== incidents_count per month ======")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nWeights:\n", coefs)


===== incidents_count per weekday =====

5-fold R^2: mean=0.018, std=0.028

Weights:
 weekday_Monday      -14.163120
weekday_Tuesday     -13.100620
weekday_Wednesday   -11.772495
weekday_Thursday     -8.999058
weekday_Sunday       -8.159533
weekday_Saturday     -1.669261
dtype: float64

====== incidents_count per month ======

5-fold R^2: mean=0.073, std=0.066

Weights:
 month_August       -4.979785
month_September    -4.773333
month_July         -1.734624
month_June         -1.440000
month_May          -1.405591
month_March        -0.534624
month_October       7.058925
month_November      8.206667
month_February     16.420993
month_December     18.271828
month_January      23.497634
dtype: float64


To confirm the finding using Linear regression, the Random Forest Regressor model was applied to the datasets

As the random forest leverages binary splits, hundred of sparse features would overfit the model, because of that, there is no relation with the `avg_phase`

The model was configured with `n_estimators=300` meaning 300 tress, which aim to ensure performance in precense of noisy dara. The next parameter, `random_state=42`, allows fair comparison between trees. `min_samples_leaf=5` estabilishes the minimum leaf size of 5 samples, reducing overfitting. Finally, the `n_jobs=-1` makes use of 1 CPU

In [11]:
from sklearn.ensemble import RandomForestRegressor

X = pd.get_dummies(merged["moon_category"], prefix="moon", dummy_na=False)
y = merged["crime_count"].fillna(0)

model = RandomForestRegressor(
    n_estimators=300, random_state=42, min_samples_leaf=5, n_jobs=-1
)
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()

print(f"\n=== Random forest on crime_count ===")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nTop drivers (importance):\n", importances)


=== Random forest on crime_count ===

5-fold R^2: mean=-0.771, std=0.796

Top drivers (importance):
 moon_Waxing Gibbous     0.091134
moon_Waning Crescent    0.094707
moon_Waning Gibbous     0.110093
moon_First Quarter      0.111520
moon_Last Quarter       0.118191
moon_Full Moon          0.144289
moon_New Moon           0.164697
moon_Waxing Crescent    0.165368
dtype: float64


In [12]:
from sklearn.ensemble import RandomForestRegressor

X = pd.get_dummies(merged_temp["weekday"], drop_first=True)
y = merged_temp["crime_count"].fillna(0)

model = RandomForestRegressor(
    n_estimators=300, random_state=42, min_samples_leaf=5, n_jobs=-1
)
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()

print(f"\n=== Random forest on crime_count per weekday ===")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nDrivers (importance):\n", importances)

X = pd.get_dummies(merged_temp["month"], drop_first=True)
y = merged_temp["crime_count"].fillna(0)

model = RandomForestRegressor(
    n_estimators=300, random_state=42, min_samples_leaf=5, n_jobs=-1
)
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()

print(f"\n=== Random forest on crime_count per month ===")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nDrivers (importance):\n", importances)


=== Random forest on crime_count per weekday ===

5-fold R^2: mean=-0.746, std=0.805

Drivers (importance):
 Monday       0.116345
Sunday       0.127903
Wednesday    0.151786
Saturday     0.166801
Tuesday      0.200146
Thursday     0.237018
dtype: float64

=== Random forest on crime_count per month ===

5-fold R^2: mean=-0.458, std=0.770

Drivers (importance):
 May          0.029641
September    0.045832
June         0.048048
November     0.049217
August       0.052979
October      0.059670
July         0.060762
December     0.077335
March        0.129593
February     0.217134
January      0.229790
dtype: float64


In [13]:
from sklearn.ensemble import RandomForestRegressor

X = pd.get_dummies(merged["moon_category"], drop_first=True)
y = merged["incidents_count"].fillna(0)

model = RandomForestRegressor(
    n_estimators=300, random_state=42, min_samples_leaf=5, n_jobs=-1
)
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()

print(f"\n=== Random forest on incidents_count ===")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nDrivers (importance):\n", importances)


=== Random forest on incidents_count ===

5-fold R^2: mean=-0.026, std=0.019

Drivers (importance):
 Waxing Crescent    0.096945
Waning Crescent    0.108895
Last Quarter       0.117286
New Moon           0.140963
Waxing Gibbous     0.161684
Full Moon          0.167035
Waning Gibbous     0.207192
dtype: float64


In [14]:
from sklearn.ensemble import RandomForestRegressor

X = pd.get_dummies(merged_temp["weekday"], drop_first=True)
y = merged_temp["incidents_count"].fillna(0)

model = RandomForestRegressor(
    n_estimators=300, random_state=42, min_samples_leaf=5, n_jobs=-1
)
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()

print(f"\n=== Random forest on incidents_count per weekday ===")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nDrivers (importance):\n", importances)

X = pd.get_dummies(merged_temp["month"], drop_first=True)
y = merged_temp["incidents_count"].fillna(0)

model = RandomForestRegressor(
    n_estimators=300, random_state=42, min_samples_leaf=5, n_jobs=-1
)
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()

print(f"\n=== Random forest on incidents_count per month ===")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nDrivers (importance):\n", importances)


=== Random forest on incidents_count per weekday ===

5-fold R^2: mean=0.018, std=0.028

Drivers (importance):
 Sunday       0.128114
Thursday     0.131605
Wednesday    0.146040
Tuesday      0.176038
Monday       0.203334
Saturday     0.214869
dtype: float64

=== Random forest on incidents_count per month ===

5-fold R^2: mean=0.073, std=0.066

Drivers (importance):
 June         0.003027
May          0.003151
July         0.003370
March        0.005106
September    0.012125
August       0.012776
October      0.071277
November     0.081086
February     0.205697
December     0.218619
January      0.383765
dtype: float64


To ensure no relation was left uncheck with the previous two models, an Possion Regressor was also applied:

the `max_iter=300` parameter ensures that the model interative optimization procedure runs a mazimum of 300 iterations. `mean_poisson_deviance` applies a loss function as Poisson models focus on deviance. `greater_is_better=False` ensures that cross-validation minimizes deviance

In [15]:
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import make_scorer, mean_poisson_deviance

X = pd.get_dummies(merged["moon_category"], drop_first=True)
y = merged["crime_count"].fillna(0)
model = PoissonRegressor(max_iter=300)
scorer = make_scorer(mean_poisson_deviance, greater_is_better=False)
dev = -cross_val_score(model, X, y, cv=5, scoring=scorer)
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n========= Poisson regression on crime_count =========")
print(f"\n5-fold mean Poisson deviance: mean={dev.mean():.3f}, std={dev.std():.3f}")
print("\nEffects:\n", coefs)


========= Poisson regression on crime_count =========

5-fold mean Poisson deviance: mean=15.777, std=6.834

Effects:
 Waxing Crescent   -0.009134
New Moon          -0.008779
Last Quarter      -0.004744
Waxing Gibbous     0.000261
Waning Crescent    0.000794
Waning Gibbous     0.000893
Full Moon          0.005164
dtype: float64


In [16]:
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import make_scorer, mean_poisson_deviance

X = pd.get_dummies(merged["moon_category"], drop_first=True)
y = merged["incidents_count"].fillna(0)
model = PoissonRegressor(max_iter=300)
scorer = make_scorer(mean_poisson_deviance, greater_is_better=False)
dev = -cross_val_score(model, X, y, cv=5, scoring=scorer)
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n========== Poisson regression on incidents_count ==========")
print(f"\n5-fold mean Poisson deviance: mean={dev.mean():.3f}, std={dev.std():.3f}")
print("\nEffects:\n", coefs)



========== Poisson regression on incidents_count ==========

5-fold mean Poisson deviance: mean=6.096, std=0.268

Effects:
 Waxing Gibbous    -0.017235
Full Moon         -0.016540
New Moon          -0.016275
Waning Crescent   -0.010239
Last Quarter      -0.009463
Waxing Crescent   -0.005591
Waning Gibbous     0.009579
dtype: float64
